# Down and Out Barrier Option
The derivative pricing is calculated based on Black Sholes assumption of constant volatility and interest rate


In [223]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
seed = 123
rng = np.random.default_rng(seed)

In [224]:
def call_payoff(S_T, K):
    return np.maximum(S_T - K, 0)

def put_payoff(S_T, K):
    return np.maximum(K - S_T, 0)

In [225]:
# The Euler discretization simulates the price path step by step
def euler_discretization(S0, r, sigma, T, dt, rng):
    N = int(T/dt)  # number of time steps
    S = np.zeros(N+1)
    S[0] = S0
    
    for t in range(1, N+1):
        error = rng.standard_normal()  # generate standard normal random variables
        S[t] = S[t-1] *(1 + r * dt + sigma * np.sqrt(dt) * error)
    return S

In [226]:
def black_scholes_call(S0, K, r, T, sigma):
    d1= (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2= d1 - sigma * np.sqrt(T)
    Nd1 = norm.cdf(d1)
    Nd2 = norm.cdf(d2)
    call_price = S0 * Nd1 - K * np.exp(-r * T) * Nd2
    return call_price

def black_scholes_put(S0, K, r, T, sigma):
    d1= (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2= d1 - sigma * np.sqrt(T)
    N_neg_d1 = norm.cdf(-d1)
    N_neg_d2 = norm.cdf(-d2)
    put_price = K * np.exp(-r * T) * N_neg_d2 - S0 * N_neg_d1
    return put_price

In [227]:
S0 = 100  # initial stock price
K = 100   # strike price for call option
r = 0.05  # risk-free interest rate
T = 1.0   # time to maturity in years
sigma = 0.2  # volatility
n_simulations = 10000 # number of Monte Carlo simulations
dt = 0.01  # time step for Euler discretization

In [228]:
S_paths = np.zeros((n_simulations, int(T/dt) + 1))
for i in range(n_simulations):
    S_path = euler_discretization(S0, r, sigma, T, dt, rng)
    S_paths[i, :] = S_path
log_S_paths = np.log(np.maximum(S_paths, 1e-12))

# compute arithmetic and geometric averages
arith_mean = np.mean(S_paths, axis =1)
geo_mean = np.exp(np.mean(log_S_paths, axis=1))
# compute payoffs for asian call, put options
discount_factor = np.exp(-r * T)
S_T = S_paths[:,-1]
bs_call = black_scholes_call(S0, K, r, T, sigma)
bs_put = black_scholes_put(S0, K, r, T, sigma)
call_payoffs = discount_factor * call_payoff(S_T, K)
put_payoffs = discount_factor * put_payoff(S_T, K)
arith_call_prices = discount_factor * call_payoff(arith_mean, K)
geo_call_prices = discount_factor * call_payoff(geo_mean, K)
arith_put_prices = discount_factor * put_payoff(arith_mean, K)
geo_put_prices = discount_factor * put_payoff(geo_mean, K)
print(f"Black Sholes call: {bs_call}, put: {bs_put}")
print(f"MC call: {np.mean(call_payoffs)}, MC put: {np.mean(put_payoffs)}")
print(f"Asian Arithmetic Call Option Price: {np.mean(arith_call_prices)}")
print(f"Asian Geometric Call Option Price: {np.mean(geo_call_prices)}")
print(f"Asian Arithmetic Put Option Price: {np.mean(arith_put_prices)}")
print(f"Asian Geometric Put Option Price: {np.mean(geo_put_prices)}")

Black Sholes call: 10.450583572185565, put: 5.573526022256971
MC call: 10.53990115275247, MC put: 5.537110559864399
Asian Arithmetic Call Option Price: 5.739789029915798
Asian Geometric Call Option Price: 5.519340733176332
Asian Arithmetic Put Option Price: 3.3283084047774056
Asian Geometric Put Option Price: 3.4462590773936186


In [229]:
# Compute SE of the estimates
arith_call_se = np.std(arith_call_prices) / np.sqrt(n_simulations)
geo_call_se = np.std(geo_call_prices) / np.sqrt(n_simulations)
arith_put_se = np.std(arith_put_prices) / np.sqrt(n_simulations)
geo_put_se = np.std(geo_put_prices) / np.sqrt(n_simulations)
print(f"Asian Arithmetic Call Option SE: {arith_call_se}")
print(f"Asian Geometric Call Option SE: {geo_call_se}")
print(f"Asian Arithmetic Put Option SE: {arith_put_se}")
print(f"Asian Geometric Put Option SE: {geo_put_se}")

Asian Arithmetic Call Option SE: 0.08069622956561767
Asian Geometric Call Option SE: 0.0777319444261994
Asian Arithmetic Put Option SE: 0.051383324406520006
Asian Geometric Put Option SE: 0.052789756224786766
